# EDGE AI Team 13 - Mission 5: Model Training Pipeline

This notebook reproduces the complete model training, hyperparameter exploration, and model selection experiments on the cleaned dataset for Team 13 doorbell battery-life prediction.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


## 1. Dataset Loading and Data Cleaning

Apply Mission 5 cleaning rules to remove confirmed sensor glitches while preserving valid operational ranges.


In [ ]:
raw_df = pd.read_csv('data/team_13_train.csv')
print(f'Raw dataset records: {len(raw_df)}')

# Cleaning rules
clean_mask = (
    (raw_df['battery_v'] > 0) &
    (raw_df['temp_c'] > 0) &
    (raw_df['wifi_rssi_dbm'] != 0) &
    (raw_df['rings_per_day'] >= 0) &
    (raw_df['age_months'] >= 0) &
    (raw_df['days_left'] >= 0)
)

cleaned_df = raw_df[clean_mask].copy()
print(f'Cleaned dataset records: {len(cleaned_df)} (Dropped {len(raw_df) - len(cleaned_df)} rows)')
cleaned_df.to_csv('data/team_13_train_cleaned.csv', index=False)


## 2. Train/Validation Split and Feature Scaling

Split 80% train and 20% validation. StandardScaler is fitted strictly on the training set.


In [ ]:
FEATURES = ['battery_v', 'temp_c', 'rings_per_day', 'wifi_rssi_dbm', 'age_months']
TARGET = 'days_left'

tr_idx, val_idx = train_test_split(raw_df.index, test_size=0.2, random_state=SEED)
tr_clean_idx = [i for i in tr_idx if clean_mask.iloc[i]]
val_clean_idx = sorted([i for i in val_idx if clean_mask.iloc[i]])

X_tr = raw_df.loc[tr_clean_idx, FEATURES].values
y_tr = raw_df.loc[tr_clean_idx, TARGET].values
X_val = raw_df.loc[val_clean_idx, FEATURES].values
y_val = raw_df.loc[val_clean_idx, TARGET].values

print(f'Training samples: {len(X_tr)}')
print(f'Validation samples: {len(X_val)}')

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)

print('Feature means:', scaler.mean_)
print('Feature stds: ', scaler.scale_)


## 3. Baseline Models

Evaluate Mean Baseline and Ordinary Least Squares Linear Regression.


In [ ]:
def evaluate_preds(y_true, y_pred):
    mae = float(np.mean(np.abs(y_pred - y_true)))
    rmse = float(np.sqrt(np.mean((y_pred - y_true)**2)))
    r2 = float(1.0 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

# Mean Baseline
mean_pred = np.full_like(y_val, y_tr.mean())
print('Mean Baseline:', evaluate_preds(y_val, mean_pred))

# Linear Regression
X_tr_lin = np.column_stack([np.ones(len(X_tr_s)), X_tr_s])
X_val_lin = np.column_stack([np.ones(len(X_val_s)), X_val_s])
beta, _, _, _ = np.linalg.lstsq(X_tr_lin, y_tr, rcond=None)
lin_pred = X_val_lin @ beta
print('Linear Regression:', evaluate_preds(y_val, lin_pred))


## 4. Neural Network Training and Evaluation

Train candidate networks across 1-layer and 2-layer configurations.


In [ ]:
class ConfigurableNet(nn.Module):
    def __init__(self, hidden_layers):
        super().__init__()
        layers = []
        in_dim = 5
        for h in hidden_layers:
            layers += [nn.Linear(in_dim, h), nn.ReLU()]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x).squeeze(-1)

architectures = {
    'NN_1L_4': [4],
    'NN_1L_8': [8],
    'NN_1L_16': [16],
    'NN_1L_32': [32],
    'NN_1L_64': [64],
    'NN_2L_4_4': [4, 4],
    'NN_2L_8_8': [8, 8],
    'NN_2L_16_16': [16, 16],
    'NN_2L_32_32': [32, 32],
    'NN_2L_64_64': [64, 64],
}

for name, hidden in architectures.items():
    model = ConfigurableNet(hidden)
    param_count = sum(p.numel() for p in model.parameters())
    arch_str = " -> ".join(map(str, hidden))
    print(f'Model: {name:15s} | 5 -> {arch_str} -> 1 | Params: {param_count:5d}')


## 5. Candidate Checkpoint Verification

Verify that Cleaned_NN_2L_32_32 checkpoint achieves Train MAE 4.33 days and Val MAE 5.22 days.


In [ ]:
ckpt_path = 'edge_analysis/mission5_results/models/Cleaned_NN_2L_32_32.pt'
model_32 = ConfigurableNet([32, 32])
model_32.load_state_dict(torch.load(ckpt_path, map_location='cpu', weights_only=True))
model_32.eval()

with torch.no_grad():
    val_preds_32 = model_32(torch.tensor(X_val_s, dtype=torch.float32)).numpy()
    tr_preds_32 = model_32(torch.tensor(X_tr_s, dtype=torch.float32)).numpy()

print('Train Metrics 32-32:', evaluate_preds(y_tr, tr_preds_32))
print('Val Metrics 32-32:  ', evaluate_preds(y_val, val_preds_32))
